# 03 · Baselines, LightGBM, LSTM and evaluation (Phases 6–9)
Models: B0 persistence · B1 utilization-only LightGBM · A0 LightGBM full telemetry · A1 + workload label ·
L0 LSTM · L1 LSTM + label (5 seeds each). Clock ablation runs only if enough jobs have clock columns.
All metrics are written by code to `work/reports/results.json` and `results_table.csv`.

In [ ]:
# --- Setup (run in Google Colab, NOT on a laptop) ---
# 1. Upload the project's `research/` folder to Google Drive at MyDrive/pred_cooling/research
# 2. Runtime > Change runtime type > GPU (needed only for notebook 03)
from google.colab import drive
drive.mount('/content/drive')
import sys, subprocess
ROOT = '/content/drive/MyDrive/pred_cooling'
sys.path.insert(0, f'{ROOT}/research/src')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lightgbm', 'pyarrow', 'onnx', 'onnxruntime', 'psutil'], check=True)
from pcool.config import Config
# work_dir holds cleaned data, features, models and reports. It stays in YOUR Drive
# (dataset licence CC BY-NC-ND 4.0: do not share the cleaned data publicly).
cfg = Config(work_dir=f'{ROOT}/work')
print('config hash', cfg.hash())

## Step 1 · Size check (reads metadata only, safe to run)
Shows how many rows must fit in RAM. The training cell prints `[mem]` lines so you can follow memory use.

In [ ]:
from pcool import pipeline
rep = pipeline.size_report(cfg)
rep

In [ ]:
!free -h
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

## Step 2 · Train and evaluate (resumable)
Every LightGBM booster and every LSTM seed is saved to `work/models/` as soon as it is trained.
If Colab disconnects: reconnect, run the setup cell, then run THIS cell again; finished parts are skipped.

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device', device, torch.cuda.get_device_name(0) if device == 'cuda' else '')
from pcool import pipeline
results = pipeline.stage_train_evaluate(cfg, device=device, run_lstm=True)

## Step 3 · Verify the outputs were written

In [ ]:
import json, os, pandas as pd, time
rp = cfg.paths()['reports']
for f in ['results.json', 'results_table.csv']:
    path = rp / f
    print(f, 'exists' if path.exists() else 'MISSING', time.ctime(path.stat().st_mtime) if path.exists() else '')
res = json.load(open(rp / 'results.json'))
print('models:', list(res['models']))
print('test set:', res['test_set'])
t = pd.read_csv(rp / 'results_table.csv')
t[t.group == 'ALL'].pivot_table(index='model', columns=['target', 'horizon_s'], values='mae').round(3)